# Solutions · 13 · Catalyst deactivation and reactor operation

Worked solutions to the exercises of [notebook 13](../notebooks/13_catalyst_deactivation_and_reactor_operation.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate, optimize
from engreact import catalysis, kinetics
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
k_d, k0, Ea, W_over_F = 0.0121, 1e8, 50e3, 5.0
conversion = lambda T, a: 1 - np.exp(-kinetics.arrhenius(k0, Ea, T) * a * W_over_F * 1e-3)

## Exercise 1

Regeneration takes 24 h and restores full activity. For a bed at constant 500 K, find the cycle length that maximises the time-averaged conversion (production during the cycle divided by cycle + regeneration time).

In [3]:
def avg_conversion(cycle_h, regen_h=24.0):
    t = np.linspace(0, cycle_h, 400)
    X = conversion(500.0, catalysis.activity(t, k_d, "second"))
    produced = (np.trapezoid(X, t) if hasattr(np, "trapezoid") else np.trapz(X, t))
    return produced / (cycle_h + regen_h)
cycles = np.linspace(20, 800, 200)
avg = [avg_conversion(c) for c in cycles]
best = cycles[int(np.argmax(avg))]
print(f"time-averaged conversion is maximised by a cycle of {best:.0f} h ({best/24:.1f} days): {max(avg):.3f} against {avg_conversion(720):.3f} for a monthly cycle")

time-averaged conversion is maximised by a cycle of 130 h (5.4 days): 0.688 against 0.470 for a monthly cycle


Short cycles waste time regenerating; long ones run a dead catalyst. The optimum here is about a week -
where the marginal conversion during the last hour of the cycle equals the average over the whole cycle plus
regeneration, the same structure as the batch-scheduling optimum of notebook 04.

## Exercise 2

Poisoning by a feed impurity gives linear decay with k_d proportional to the impurity concentration. If a guard bed removes 90 % of the impurity, how much longer is the cycle? Compare with the cost of 10 x more catalyst.

In [4]:
k_lin = 0.004                                                       # 1/h, linear poisoning at the current impurity level
end = lambda k_: 1 / k_ * (1 - 0.3)                                 # time for activity to fall to 0.3 under linear decay
print(f"cycle to activity 0.3: {end(k_lin):.0f} h now; with a guard bed removing 90 % of the poison: {end(0.1 * k_lin):.0f} h - ten times longer")
print("ten times more catalyst would hold conversion for about as long only if the poison were spread evenly; in a fixed bed it deposits at the inlet, so extra catalyst buys less than the guard bed does, and costs more")

cycle to activity 0.3: 175 h now; with a guard bed removing 90 % of the poison: 1750 h - ten times longer
ten times more catalyst would hold conversion for about as long only if the poison were spread evenly; in a fixed bed it deposits at the inlet, so extra catalyst buys less than the guard bed does, and costs more


Linear poisoning is proportional to the poison arriving, so cutting the poison tenfold lengthens the cycle
tenfold; a guard bed is cheap catalyst (or an adsorbent) sacrificed to protect expensive catalyst. Extra
main-bed catalyst is the worse investment, because poisons deposit as a front from the inlet rather than
uniformly.

## Exercise 3

**Implement it yourself:** integrate the activity ODE da/dt = -k_d(T) a^2 along the library's temperature-time trajectory and confirm that the product k(T) a stays constant.

In [5]:
traj = catalysis.temperature_time_trajectory(k_d0=2.0, E_d=60e3, k0=1e8, Ea=50e3, T0=500.0, t_max=200 * 86400, T_max=560.0)
T_of_t = lambda t: np.interp(t, traj["t"], traj["T"])
sol = integrate.solve_ivp(lambda t, a: [-kinetics.arrhenius(2.0, 60e3, T_of_t(t)) * a[0] ** 1], (0, traj["t"][-1]), [1.0], t_eval=traj["t"], rtol=1e-9)
product = kinetics.arrhenius(1e8, 50e3, traj["T"]) * sol.y[0]
print(f"k(T) a along the trajectory: min {product.min():.4f}, max {product.max():.4f} (target k(T0) = {kinetics.arrhenius(1e8, 50e3, 500.0):.4f}) - constant, as designed")

k(T) a along the trajectory: min 596.3936, max 597.9130 (target k(T0) = 597.9130) - constant, as designed


Re-integrating the decay along the library's temperature schedule and multiplying by the Arrhenius factor
gives a constant product: the schedule does what it claims. (The library's decay is first order in activity;
the exercise's $a^2$ would need its own schedule.)